# Fraud Detection: Model Evaluation

This notebook trains CatBoost and XGBoost on the engineered feature set and evaluates both models with per-class F1, confusion matrices, feature importance, and learning curves.

**Prerequisites:** Run `data_preprocessing.ipynb` first to generate `outputs/train_with_new_features.csv` and `outputs/test_with_new_features.csv`.

Importing libraries and adding constraints

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import catboost as cb
from catboost import CatBoostClassifier, Pool
from xgboost import XGBClassifier
from sklearn.metrics import (
    f1_score,
    classification_report,
    confusion_matrix
)

sns.set_theme(style='darkgrid')
plt.rcParams['figure.dpi'] = 120

# Constants 
TRAIN_PATH = 'C:/Users/Ilike/OneDrive/Year 3/Personal Projects/Fraud Project/outputs/train_with_new_features.csv'
TEST_PATH  = 'C:/Users/Ilike/OneDrive/Year 3/Personal Projects/Fraud Project/outputs/test_with_new_features.csv'

TARGET    = 'urgency_level'
ID_COL    = 'id'
TIME_COL  = 'step'
CAT_COLS  = ['type']
DROP_COLS = ['nameOrig', 'nameDest']

CLASS_NAMES = ['No Action', 'Monitor', 'Review', 'Immediate Action']
COLORS = ['#4CAF50', '#FFC107', '#FF9800', '#F44336']

print('Setup complete.')

Load and Split Data

In [ ]:
# Load feature-engineered data produced by data_preprocessing.ipynb
train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

train.columns = train.columns.str.strip()
test.columns  = test.columns.str.strip()

train['type'] = train['type'].astype(str)
test['type']  = test['type'].astype(str)

# Drop raw name columns, already captured by dest_is_merchant flag
train = train.drop(columns=[c for c in DROP_COLS if c in train.columns])
test  = test.drop(columns=[c for c in DROP_COLS if c in test.columns])

# Time-based split
# Last 20% of time steps = validation
# Simulates real deployment: train on historical data, evaluate on recent transactions
cutoff  = np.quantile(train[TIME_COL], 0.80)
tr_mask = train[TIME_COL] <= cutoff
va_mask = train[TIME_COL] > cutoff

train_tr = train.loc[tr_mask].copy()
train_va = train.loc[va_mask].copy()

y_tr = train_tr[TARGET].astype(int)
y_va = train_va[TARGET].astype(int)

# Inverse-frequency class weights 
# Rare fraud classes get upweighted so the model doesn't ignore them
counts  = y_tr.value_counts().sort_index()
total   = counts.sum()
class_w = (total / (len(counts) * counts)).to_dict()
w_tr    = y_tr.map(class_w).astype(float).values

print(f'Train rows: {len(train_tr):,}  |  Val rows: {len(train_va):,}')
print(f'Class weights: { {k: round(v,1) for k,v in class_w.items()} }')
print('\nVal urgency distribution:')
print(y_va.value_counts().sort_index().to_string())

Train CatBoost

In [ ]:
# CatBoost handles the type column natively, so no one-hot encoding needed
X_tr_cb   = train_tr.drop(columns=[TARGET, ID_COL])
X_va_cb   = train_va.drop(columns=[TARGET, ID_COL])
X_test_cb = test.drop(columns=[ID_COL])

# CatBoost needs the column INDEX of categorical features, not the name
cat_idx = [X_tr_cb.columns.get_loc('type')]

# Pool bundles features, labels, weights, and categorical info into one object
train_pool = cb.Pool(X_tr_cb, y_tr, cat_features=cat_idx, weight=w_tr)
valid_pool = cb.Pool(X_va_cb, y_va, cat_features=cat_idx)
test_pool  = cb.Pool(X_test_cb, cat_features=cat_idx)

cb_model = CatBoostClassifier(
    loss_function='MultiClass',
    eval_metric='TotalF1:average=Macro;use_weights=False',
    iterations=8000,
    learning_rate=0.05,
    depth=9,
    l2_leaf_reg=5.0,
    random_seed=42,
    class_weights=[class_w[i] for i in range(4)],
    od_type='Iter',    # early stopping method
    od_wait=200,       # stop if no improvement after 200 rounds
    verbose=500
)

cb_model.fit(train_pool, eval_set=valid_pool, use_best_model=True)

# Predict on validation set
cb_val_pred = cb_model.predict(X_va_cb).astype(int).reshape(-1)
cb_macro_f1 = f1_score(y_va, cb_val_pred, average='macro')
print(f'\nCatBoost Validation Macro F1: {cb_macro_f1:.4f}')
print(f'Best iteration: {cb_model.get_best_iteration()}')

Train XGBoost

In [ ]:
# XGBoost requires one-hot encoding for categorical features
X_tr_xgb   = train_tr.drop(columns=[TARGET, ID_COL])
X_va_xgb   = train_va.drop(columns=[TARGET, ID_COL])
X_test_xgb = test.drop(columns=[ID_COL])

# One-hot encode the type column; dummy_na=True handles unseen values in test
X_tr_xgb   = pd.get_dummies(X_tr_xgb,   columns=CAT_COLS, dummy_na=True)
X_va_xgb   = pd.get_dummies(X_va_xgb,   columns=CAT_COLS, dummy_na=True)
X_test_xgb = pd.get_dummies(X_test_xgb, columns=CAT_COLS, dummy_na=True)

# Align columns so all splits are identical
# join='left' keeps X_tr_xgb's columns as the reference; missing cols filled with 0
X_tr_xgb, X_va_xgb   = X_tr_xgb.align(X_va_xgb,   join='left', axis=1, fill_value=0)
X_tr_xgb, X_test_xgb = X_tr_xgb.align(X_test_xgb, join='left', axis=1, fill_value=0)

xgb_model = XGBClassifier(
    objective='multi:softprob',
    num_class=4,
    n_estimators=3000,
    learning_rate=0.05,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.7,
    reg_lambda=5.0,
    min_child_weight=5,
    gamma=0.1,
    tree_method='hist',
    random_state=42,
    eval_metric='mlogloss',
    early_stopping_rounds=150,  # stop if no improvement for 150 rounds
)

xgb_model.fit(
    X_tr_xgb, y_tr,
    sample_weight=w_tr,
    eval_set=[(X_va_xgb, y_va)],
    verbose=500,
)

xgb_val_pred = xgb_model.predict_proba(X_va_xgb).argmax(axis=1)
xgb_macro_f1 = f1_score(y_va, xgb_val_pred, average='macro')
print(f'\nXGBoost Validation Macro F1: {xgb_macro_f1:.4f}')
print(f'Best iteration: {xgb_model.best_iteration}')

Per-Class F1 Breakdown

Macro F1 tells us the average performance across all classes, but per-class F1 shows exactly how well the model performs on each urgency level individually, which is especially important for the rare fraud classes.

In [ ]:
# Print full classification reports for both models
print('=' * 55)
print('CatBoost — Classification Report')
print('=' * 55)
print(classification_report(y_va, cb_val_pred, target_names=CLASS_NAMES))

print('=' * 55)
print('XGBoost — Classification Report')
print('=' * 55)
print(classification_report(y_va, xgb_val_pred, target_names=CLASS_NAMES))

# Side-by-side per-class F1 bar chart
# Extract per-class F1 scores for each model
cb_f1_per_class  = f1_score(y_va, cb_val_pred,  average=None)
xgb_f1_per_class = f1_score(y_va, xgb_val_pred, average=None)

x = np.arange(len(CLASS_NAMES))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 5))

# Plot two sets of bars side by side — one per model
bars_cb  = ax.bar(x - width/2, cb_f1_per_class,  width, label='CatBoost',  color='#2196F3', alpha=0.85, edgecolor='white')
bars_xgb = ax.bar(x + width/2, xgb_f1_per_class, width, label='XGBoost',   color='#FF9800', alpha=0.85, edgecolor='white')

# Add value labels on top of each bar
for bar in bars_cb:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for bar in bars_xgb:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES)
ax.set_ylabel('F1 Score')
ax.set_ylim(0, 1.1)
ax.set_title('Per-Class F1 Score: CatBoost vs XGBoost', fontsize=13, fontweight='bold')
ax.legend()
ax.axhline(0.5, color='grey', linestyle='--', linewidth=0.8, alpha=0.6)  # reference line

plt.tight_layout()
plt.savefig('outputs/plot_per_class_f1.png', bbox_inches='tight')
plt.show()

print(f'\nCatBoost  Macro F1: {cb_macro_f1:.4f}')
print(f'XGBoost   Macro F1: {xgb_macro_f1:.4f}')

Confusion Matrices

A confusion matrix shows exactly which classes the model confuses with each other. Each row is the true class, each column is the predicted class. Diagonal cells are correct predictions; off-diagonal cells are errors.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, preds, title in zip(
    axes,
    [cb_val_pred, xgb_val_pred],
    ['CatBoost', 'XGBoost']
):
    cm = confusion_matrix(y_va, preds)

    # Normalise each row so values show the proportion of each true class
    # that was predicted correctly vs incorrectly
    # This is more useful than raw counts when classes are imbalanced
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    sns.heatmap(
        cm_norm,
        annot=True,
        fmt='.2f',
        cmap='Blues',
        xticklabels=CLASS_NAMES,
        yticklabels=CLASS_NAMES,
        linewidths=0.5,
        linecolor='white',
        vmin=0, vmax=1,
        ax=ax
    )
    ax.set_title(f'{title} — Normalised Confusion Matrix', fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')
    ax.tick_params(axis='x', rotation=15)
    ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.savefig('C:/Users/Ilike/OneDrive/Year 3/Personal Projects/Fraud Project/outputs/plot_confusion_matrices.png', bbox_inches='tight')
plt.show()

Feature Importance

Feature importance shows which features each model relied on most when making predictions. This helps validate that the engineered features (deltaOrg, check_Amount_Size etc.) are actually contributing useful signal.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# CatBoost feature importance 
# get_feature_importance() returns importance scores for each feature
cb_importances = pd.Series(
    cb_model.get_feature_importance(),
    index=X_tr_cb.columns
).sort_values(ascending=True).tail(15)  # top 15 features

axes[0].barh(cb_importances.index, cb_importances.values,
             color='#2196F3', alpha=0.85, edgecolor='white')
axes[0].set_title('CatBoost — Top 15 Feature Importances', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Importance Score')

# XGBoost feature importance 
# feature_importances_ uses gain by default, so how much each feature improves splits
xgb_importances = pd.Series(
    xgb_model.feature_importances_,
    index=X_tr_xgb.columns
).sort_values(ascending=True).tail(15)

axes[1].barh(xgb_importances.index, xgb_importances.values,
             color='#FF9800', alpha=0.85, edgecolor='white')
axes[1].set_title('XGBoost — Top 15 Feature Importances', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Importance Score')

plt.tight_layout()
plt.savefig('C:/Users/Ilike/OneDrive/Year 3/Personal Projects/Fraud Project/outputs/plot_feature_importance.png', bbox_inches='tight')
plt.show()

# Save to CSV for reference
pd.Series(cb_model.get_feature_importance(), index=X_tr_cb.columns
         ).sort_values(ascending=False).to_csv('C:/Users/Ilike/OneDrive/Year 3/Personal Projects/Fraud Project/outputs/cb_feature_importances.csv', header=['importance'])
pd.Series(xgb_model.feature_importances_, index=X_tr_xgb.columns
         ).sort_values(ascending=False).to_csv('C:/Users/Ilike/OneDrive/Year 3/Personal Projects/Fraud Project/outputs/xgb_feature_importances.csv', header=['importance'])
print('Feature importances saved to outputs/')

Learning Curves

Learning curves plot how the validation score changes as training progresses. A healthy learning curve shows the validation score improving and then plateauing, if it starts dropping, the model is overfitting.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# CatBoost learning curve 
# get_evals_result() returns the metric value at each iteration
cb_evals = cb_model.get_evals_result()

# The key structure is: {'learn': {metric: [values]}, 'validation': {metric: [values]}}
cb_val_scores = cb_evals.get('validation', {})
if cb_val_scores:
    metric_key = list(cb_val_scores.keys())[0]
    cb_scores = cb_val_scores[metric_key]
    axes[0].plot(cb_scores, color='#2196F3', linewidth=1.5, label='Validation')

    # Mark the best iteration with a vertical dashed line
    best_iter = cb_model.get_best_iteration()
    axes[0].axvline(best_iter, color='red', linestyle='--', linewidth=1,
                    label=f'Best iteration: {best_iter}')

axes[0].set_title('CatBoost — Validation Score Over Iterations', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Macro F1')
axes[0].legend()

# XGBoost learning curve 
# evals_result_ is populated when eval_set is passed during fit()
xgb_evals = xgb_model.evals_result()

# XGBoost stores results as: {'validation_0': {'mlogloss': [values]}}
xgb_scores = xgb_evals['validation_0']['mlogloss']
axes[1].plot(xgb_scores, color='#FF9800', linewidth=1.5, label='Validation mlogloss')

best_iter_xgb = xgb_model.best_iteration
axes[1].axvline(best_iter_xgb, color='red', linestyle='--', linewidth=1,
                label=f'Best iteration: {best_iter_xgb}')

axes[1].set_title('XGBoost — Validation Loss Over Iterations', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Log Loss')
axes[1].legend()

plt.tight_layout()
plt.savefig('C:/Users/Ilike/OneDrive/Year 3/Personal Projects/Fraud Project/outputs/plot_learning_curves.png', bbox_inches='tight')
plt.show()

Results Summary and Submission

In [ ]:
# Results summary table
cb_f1_per_class  = f1_score(y_va, cb_val_pred,  average=None)
xgb_f1_per_class = f1_score(y_va, xgb_val_pred, average=None)

results = pd.DataFrame({
    'Model': ['CatBoost', 'XGBoost'],
    'Macro F1': [round(cb_macro_f1, 4), round(xgb_macro_f1, 4)],
    'F1 — No Action':        [round(cb_f1_per_class[0], 4), round(xgb_f1_per_class[0], 4)],
    'F1 — Monitor':          [round(cb_f1_per_class[1], 4), round(xgb_f1_per_class[1], 4)],
    'F1 — Review':           [round(cb_f1_per_class[2], 4), round(xgb_f1_per_class[2], 4)],
    'F1 — Immediate Action': [round(cb_f1_per_class[3], 4), round(xgb_f1_per_class[3], 4)],
})

print('\n=== Final Results ===')
print(results.to_string(index=False))

# Pick the best model and generate submission 
# Use whichever model has higher Macro F1
if cb_macro_f1 >= xgb_macro_f1:
    print(f'\nBest model: CatBoost (Macro F1: {cb_macro_f1:.4f})')
    test_pred = cb_model.predict(test_pool).astype(int).reshape(-1)
else:
    print(f'\nBest model: XGBoost (Macro F1: {xgb_macro_f1:.4f})')
    test_pred = xgb_model.predict_proba(X_test_xgb).argmax(axis=1)

submission = pd.DataFrame({'id': test[ID_COL], TARGET: test_pred})
submission.to_csv('C:/Users/Ilike/OneDrive/Year 3/Personal Projects/Fraud Project/outputs/submission.csv', index=False)
print(f'Submission saved to outputs/submission.csv ({len(submission):,} rows)')
print(submission[TARGET].value_counts().sort_index())